# 🚗 Y4GA — Third Notebook
## Arquitectura OOP Completa: Dataset Maestro

**Flujo de ejecución:**

```
PASO 1 → UberPDFDownloader   → Descarga PDFs del portal Uber  → Carpeta Descargas_Uber_{Mes}_{Año}/
PASO 2 → UberActivityBot     → Extrae viajes individuales      → viajes_uber_detallados.json
PASO 3 → UberReportExtractor → Parsea UN pdf semanal           → dict con datos de la semana
PASO 4 → TripActivity        → Parsea UN viaje del JSON        → dict con datos del viaje
PASO 5 → UberDatasetManager  → Orquesta y fusiona todo         → df_weekly + df_trips
PASO 6 → DataExporter        → Guarda CSVs limpios             → dataset_output/
```

| Clase | Responsabilidad |
|---|---|
| `UberPDFDownloader` | Bot que descarga PDFs del portal Uber por mes/año |
| `UberActivityBot` | Bot que intercepta viajes individuales del portal |
| `UberReportExtractor` | Parsea UN pdf semanal → datos financieros |
| `TripActivity` | Parsea UN viaje del JSON → datos del viaje |
| `UberDatasetManager` | Orquesta ambas fuentes, calcula columnas derivadas |
| `DataExporter` | Guarda los CSVs finales con validación |

---
## CELDA 1 — Imports

In [5]:
import pdfplumber
import re
import os
import glob
import json
import asyncio
import random
import pandas as pd
from datetime import datetime, timezone
from playwright.async_api import async_playwright
from playwright.sync_api import sync_playwright

print('✅ Imports OK')

✅ Imports OK


---
## CELDA 2 — Clase `UberPDFDownloader`
**Responsabilidad única:** Abre el portal de Uber, navega a la sección de estados de cuenta
y descarga todos los PDFs de un mes/año específico a una carpeta local.

**Cómo usarlo:**
```python
bot = UberPDFDownloader(session_dir='uber_session')
await bot.run()  # Pide mes/año, descarga y crea la carpeta automáticamente
```

In [6]:
bot = UberPDFDownloader(session_dir='uber_session')
await bot.run() 


🤖 UberPDFDownloader → Diciembre 2024
   Destino: Uber_Reports/Diciembre_2024/
-------------------------------------------------------
🛠️  Selecciona el año "2024" y el mes "diciembre" en la página.
   Espera a que aparezcan los botones "Descargar PDF".

🚀 Descargando...
     ✅ resumen_2 de dic, 4_00.pdf
     ✅ resumen_9 de dic, 4_00.pdf
     ✅ resumen_16 de dic, 4_00.pdf
     ✅ resumen_23 de dic, 4_00.pdf
     ✅ resumen_30 de dic, 4_00.pdf
     ✅ resumen_6 de ene, 4_00.pdf

✨ 6 PDFs guardados → Uber_Reports/Diciembre_2024/


In [3]:
# =========================================================================
# ⚠️ AVISO LEGAL / DISCLAIMER
# =========================================================================
# Herramienta de descarga de datos personales para uso estrictamente
# individual. El autor no se responsabiliza del mal uso o pérdida de acceso.
# Úselo bajo su propio riesgo y discreción.
# =========================================================================

class UberPDFDownloader:
    """
    Bot que descarga PDFs de resúmenes semanales del portal Uber.

    Estructura de carpetas que genera:
        Uber_Reports/
        ├── Diciembre_2024/
        │   ├── resumen_sem1.pdf
        │   └── resumen_sem2.pdf
        ├── Enero_2025/
        └── Abril_2025/

    Uso — descargar UN mes:
        downloader = UberPDFDownloader()
        await downloader.run()

    Uso — descargar VARIOS meses en loop:
        downloader = UberPDFDownloader()
        meses = [('diciembre', '2024'), ('enero', '2025'), ('abril', '2025')]
        for mes, anio in meses:
            await downloader.run(mes=mes, anio=anio)
    """

    # Carpeta raíz donde se guardan TODOS los meses
    ROOT_FOLDER    = 'Uber_Reports'
    SESSION_DIR    = 'uber_session'
    URL_STATEMENTS = 'https://drivers.uber.com/earnings/statements'

    def __init__(self, session_dir: str = 'uber_session'):
        self.session_dir    = session_dir
        self.output_folder  = None   # Se define en run(): Uber_Reports/Mes_Año/
        self.downloads_count = 0
        os.makedirs(self.ROOT_FOLDER, exist_ok=True)
        os.makedirs(self.session_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # UTILIDADES PRIVADAS
    # ------------------------------------------------------------------

    def _build_output_folder(self, mes: str, anio: str) -> str:
        """
        Construye la ruta de la subcarpeta del mes.
        Ej: 'Uber_Reports/Abril_2025/'
        """
        nombre = f'{mes.capitalize()}_{anio}'
        path   = os.path.join(self.ROOT_FOLDER, nombre)
        os.makedirs(path, exist_ok=True)
        return path

    def _pedir_mes_anio(self) -> tuple[str, str]:
        """Pide mes y año por consola. Devuelve (mes, anio)."""
        entrada = input('👉 ¿Qué mes y año vas a descargar? (ej: abril 2025): ').strip().lower()
        partes  = entrada.split()
        if len(partes) < 2:
            raise ValueError('❌ Formato incorrecto. Escribe: mes año  (ej: abril 2025)')
        return partes[0], partes[1]

    async def _descargar_pdfs_del_mes(self, page, mes: str, anio: str) -> int:
        """
        Barre la tabla del portal y descarga los PDFs del mes indicado.
        Devuelve la cantidad de archivos descargados.
        """
        mes_abreviado = mes[:3]   # 'abril' → 'abr'
        self.output_folder = self._build_output_folder(mes, anio)
        descargas = 0

        filas = await page.locator('tr').all()

        for fila in filas:
            texto_fila = (await fila.inner_text()).lower()

            if mes_abreviado in texto_fila and 'descargar pdf' in texto_fila:
                boton = fila.get_by_role('button', name='Descargar PDF')

                if await boton.is_visible():
                    await boton.scroll_into_view_if_needed()

                    async with page.expect_download() as dl_info:
                        await boton.click()

                    download   = await dl_info.value
                    ruta_final = os.path.join(self.output_folder, download.suggested_filename)
                    await download.save_as(ruta_final)

                    print(f'     ✅ {download.suggested_filename}')
                    descargas += 1
                    await asyncio.sleep(random.uniform(0.5, 1.5))

        return descargas

    # ------------------------------------------------------------------
    # MÉTODO PRINCIPAL
    # ------------------------------------------------------------------

    async def run(self, mes: str = None, anio: str = None):
        """
        Descarga los PDFs de un mes/año.
        Si mes y anio son None, los pide por consola.
        Puede llamarse múltiples veces con el mismo contexto de browser.
        """
        if mes is None or anio is None:
            mes, anio = self._pedir_mes_anio()

        print(f'\n🤖 UberPDFDownloader → {mes.capitalize()} {anio}')
        print(f'   Destino: {self.ROOT_FOLDER}/{mes.capitalize()}_{anio}/')
        print('-' * 55)

        async with async_playwright() as p:
            context = await p.chromium.launch_persistent_context(
                os.path.abspath(self.session_dir),
                headless=False,
                args=['--no-sandbox', '--disable-dev-shm-usage']
            )
            page = context.pages[0]
            await page.goto(self.URL_STATEMENTS, wait_until='networkidle')

            print(f'🛠️  Selecciona el año "{anio}" y el mes "{mes}" en la página.')
            print('   Espera a que aparezcan los botones "Descargar PDF".')
            input('\n⏳ Cuando estés listo presiona ENTER...')

            print(f'\n🚀 Descargando...')
            try:
                n = await self._descargar_pdfs_del_mes(page, mes, anio)
                self.downloads_count += n
                print(f'\n✨ {n} PDFs guardados → {self.output_folder}/')
            except Exception as e:
                print(f'❌ Error: {e}')

            await asyncio.sleep(1)
            await context.close()


In [10]:
bot = UberPDFDownloader(session_dir='uber_session')
await bot.run() 


🤖 UberPDFDownloader → Enero 2025
   Destino: Uber_Reports/Enero_2025/
-------------------------------------------------------
🛠️  Selecciona el año "2025" y el mes "enero" en la página.
   Espera a que aparezcan los botones "Descargar PDF".

🚀 Descargando...
     ✅ resumen_6 de ene, 4_00.pdf
     ✅ resumen_13 de ene, 4_00.pdf
     ✅ resumen_20 de ene, 4_00.pdf
     ✅ resumen_27 de ene, 4_00.pdf
     ✅ resumen_3 de feb, 4_00.pdf

✨ 5 PDFs guardados → Uber_Reports/Enero_2025/


In [23]:
bot = UberPDFDownloader(session_dir='uber_session')
await bot.run() 


🤖 UberPDFDownloader → Diciembre 2025
   Destino: Uber_Reports/Diciembre_2025/
-------------------------------------------------------
🛠️  Selecciona el año "2025" y el mes "diciembre" en la página.
   Espera a que aparezcan los botones "Descargar PDF".

🚀 Descargando...
     ✅ resumen_1 de dic, 4_00.pdf
     ✅ resumen_8 de dic, 4_00.pdf
     ✅ resumen_15 de dic, 4_00.pdf
     ✅ resumen_22 de dic, 4_00.pdf
     ✅ resumen_29 de dic, 4_00.pdf
     ✅ resumen_5 de ene, 4_00.pdf

✨ 6 PDFs guardados → Uber_Reports/Diciembre_2025/


In [2]:
import asyncio
import os
import json
import pandas as pd
from playwright.async_api import async_playwright  # <--- CAMBIO CLAVE

class UberActivityBot:
    SESSION_FILE = 'uber_session/gs_session.json'
    URL_ACTIVITIES = 'https://drivers.uber.com/earnings/activities'
    OUTPUT_JSON = 'viajes_uber_detallados.json'

    def __init__(self, session_file: str = 'uber_session/gs_session.json'):
        self.session_file = session_file
        self._viajes_acumulados = []
        os.makedirs('uber_session', exist_ok=True)

    async def _interceptar_trafico(self, response):
        if 'getWebActivityFeed' in response.url and response.status == 200:
            try:
                datos = await response.json()
                nuevos = datos.get('data', {}).get('activities', [])
                if nuevos:
                    self._viajes_acumulados.extend(nuevos)
                    print(f'  ✅ Capturados {len(nuevos)} viajes (Total: {len(self._viajes_acumulados)})')
            except:
                pass

    async def run(self, stop_year=2024):
        print('🤖 UberActivityBot iniciado (Modo Async) - Navegación 2025')
        
        async with async_playwright() as p:
            # Lanzamos el navegador
            browser = await p.chromium.launch(headless=False)
            
            # Cargar sesión si existe
            if os.path.exists(self.session_file):
                context = await browser.new_context(storage_state=self.session_file)
            else:
                context = await browser.new_context()

            page = await context.new_page()
            
            # Configurar intercepción
            page.on('response', self._interceptar_trafico)
            
            await page.goto(self.URL_ACTIVITIES)

            print('\n🛠️ Si no has iniciado sesión, hazlo ahora en la ventana del navegador.')
            input('⏳ Cuando veas tus viajes, presiona ENTER aquí para empezar la automatización...')

            keep_going = True
            while keep_going:
                try:
                    # Le damos 60 segundos y que no sea estricto
                    await page.wait_for_selector('div[data-testid="date-range-selector"]', timeout=60000)
                except Exception:
                    print("⚠️ El selector de fecha no apareció. ¿Te sacó Uber de la sesión?")
                    input("👉 Por favor, acomoda la página manualmente y presiona ENTER para continuar...")

                date_text = await page.locator('div[data-testid="date-range-selector"]').inner_text()
                
                print(f'📅 Periodo actual: {date_text}')

                # Condición de parada (Si llegamos a 2024 y ya no hay rastro de 2025)
                if str(stop_year) in date_text and "2025" not in date_text:
                    print(f"🏁 Meta alcanzada ({stop_year}). Deteniendo...")
                    break

                # Pequeña espera para asegurar captura
                await asyncio.sleep(2)

                # Clic en "Semana anterior" (flecha izquierda)
                # Buscamos el botón que tiene el icono de flecha atrás
                prev_button = page.locator('button').filter(has=page.locator('svg')).first
                
                if await prev_button.is_visible():
                    await prev_button.click()
                    await page.wait_for_load_state('networkidle')
                else:
                    print("⚠️ No se encontró el botón para retroceder.")
                    break

            # Guardar resultados al final
            await context.storage_state(path=self.session_file)
            self._guardar_json()
            await browser.close()

    def _guardar_json(self):
        if self._viajes_acumulados:
            # Eliminar duplicados
            df = pd.DataFrame(self._viajes_acumulados).drop_duplicates(subset=['uuid'])
            df.to_json(self.OUTPUT_JSON, orient='records', force_ascii=False, indent=4)
            print(f'\n🏆 {len(df)} viajes guardados en {self.OUTPUT_JSON}')

In [4]:
# Instanciar el bot
bot = UberActivityBot()

# Ejecutar (esto abrirá el navegador y hará los clics por ti)
await bot.run(stop_year=2024)

🤖 UberActivityBot iniciado (Modo Async) - Navegación 2025

🛠️ Si no has iniciado sesión, hazlo ahora en la ventana del navegador.


⏳ Cuando veas tus viajes, presiona ENTER aquí para empezar la automatización... 


  ✅ Capturados 15 viajes (Total: 15)
  ✅ Capturados 15 viajes (Total: 30)
  ✅ Capturados 15 viajes (Total: 45)
  ✅ Capturados 15 viajes (Total: 60)
  ✅ Capturados 14 viajes (Total: 74)
  ✅ Capturados 6 viajes (Total: 80)
⚠️ El selector de fecha no apareció. ¿Te sacó Uber de la sesión?


👉 Por favor, acomoda la página manualmente y presiona ENTER para continuar... 


TargetClosedError: Locator.inner_text: Target page, context or browser has been closed

In [14]:
import os

for root, dirs, files in os.walk('.'):
    # Ignorar carpetas ocultas y uber_session
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != 'uber_session']
    nivel = root.replace('.', '').count(os.sep)
    indent = '    ' * nivel
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        if f.endswith('.pdf'):
            print(f'{indent}    {f}')

./
    pdf_uber/
        resumen_16_23_dic_2024.pdf
        resumen_23_30_dic.pdf
        resumen_30dic_6ene.pdf
        resumen_6_13_enero.pdf
        resumen_20_27_enero_2025.pdf
        resumen_13_20_enero.pdf
        resumen_27enero_3feb.pdf
        resumen_3-10_feb.pdf
        resumen_10_17_feb.pdf
        resumen_17_24_feb.pdf
    Downloads_Uber_Diciembre_2024/
        resumen_2 de dic, 4_00.pdf
        resumen_9 de dic, 4_00.pdf
        resumen_16 de dic, 4_00.pdf
        resumen_23 de dic, 4_00.pdf
        resumen_30 de dic, 4_00.pdf
        resumen_6 de ene, 4_00.pdf
    Descargas_Uber_Enero_2024/
        resumen_1 de ene, 4_00.pdf
        resumen_8 de ene, 4_00.pdf
        resumen_15 de ene, 4_00.pdf
        resumen_22 de ene, 4_00.pdf
        resumen_29 de ene, 4_00.pdf
        resumen_5 de feb, 4_00.pdf
    Descargas_Uber_Marzo_2025/
        resumen_3 de mar, 4_00.pdf
        resumen_10 de mar, 4_00.pdf
        resumen_17 de mar, 4_00.pdf
        resumen_24 de mar, 4_00.pdf

In [15]:
import os, glob

# Ver TODOS los PDFs y dónde están
todos = glob.glob('./**/*.pdf', recursive=True)
for p in sorted(todos):
    print(p)

./Descargas_Uber_Abril_2025/resumen_14 de abr, 4_00.pdf
./Descargas_Uber_Abril_2025/resumen_21 de abr, 4_00.pdf
./Descargas_Uber_Abril_2025/resumen_28 de abr, 4_00.pdf
./Descargas_Uber_Abril_2025/resumen_5 de may, 4_00.pdf
./Descargas_Uber_Abril_2025/resumen_7 de abr, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_1 de ene, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_15 de ene, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_22 de ene, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_29 de ene, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_5 de feb, 4_00.pdf
./Descargas_Uber_Enero_2024/resumen_8 de ene, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_10 de mar, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_17 de mar, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_24 de mar, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_3 de mar, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_31 de mar, 4_00.pdf
./Descargas_Uber_Marzo_2025/resumen_7 de abr, 4_00.pdf
./Downloads_Uber_Diciembre_2024/resumen_16 de dic, 4_00

---
## CELDA 3 — Clase `UberActivityBot`
**Responsabilidad única:** Intercepta el tráfico de red del portal Uber y captura
los viajes individuales en formato JSON.

**Cómo usarlo:**
```python
bot = UberActivityBot()
bot.run()  # Abre el browser, captura viajes, guarda JSON al presionar ENTER
```

In [5]:
class UberActivityBot:
    """
    Bot que intercepta el endpoint getWebActivityFeed de Uber
    y guarda los viajes individuales en JSON.

    Uso:
        bot = UberActivityBot()
        bot.run()
    """

    SESSION_FILE = 'uber_session/gs_session.json'
    URL_ACTIVITIES = 'https://drivers.uber.com/earnings/activities'
    OUTPUT_JSON = 'viajes_uber_detallados.json'
    OUTPUT_CSV  = 'viajes_uber_bruto.csv'

    def __init__(self, session_file: str = 'uber_session/gs_session.json'):
        self.session_file = session_file
        self._viajes_acumulados = []
        os.makedirs('uber_session', exist_ok=True)

    def _interceptar_trafico(self, response):
        """Callback que se ejecuta en cada respuesta HTTP del browser."""
        if 'getWebActivityFeed' in response.url and response.status == 200:
            try:
                datos = response.json()
                nuevos = datos.get('data', {}).get('activities', [])
                if nuevos:
                    self._viajes_acumulados.extend(nuevos)
                    print(f'  ✅ Capturados {len(nuevos)} viajes (Total: {len(self._viajes_acumulados)})')
            except:
                pass

    def _guardar_resultados(self):
        """Guarda los viajes capturados en JSON y CSV."""
        if not self._viajes_acumulados:
            print('⚠️  No se capturaron viajes.')
            return

        with open(self.OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(self._viajes_acumulados, f, ensure_ascii=False, indent=4)

        pd.DataFrame(self._viajes_acumulados).to_csv(self.OUTPUT_CSV, index=False)

        print(f'\n🏆 {len(self._viajes_acumulados)} viajes guardados:')
        print(f'   → {self.OUTPUT_JSON}')
        print(f'   → {self.OUTPUT_CSV}')

    def run(self):
        """Punto de entrada principal. Abre el browser y espera ENTER para guardar."""
        print('🤖 UberActivityBot iniciado')
        print('-' * 55)

        with sync_playwright() as p:
            browser = p.chromium.launch(headless=False)

            if os.path.exists(self.session_file):
                print('🔐 Sesión existente cargada.')
                context = browser.new_context(storage_state=self.session_file)
            else:
                print('🆕 Sin sesión previa — deberás loguearte esta vez.')
                context = browser.new_context()

            page = context.new_page()
            page.on('response', self._interceptar_trafico)
            page.goto(self.URL_ACTIVITIES)

            print('\n💡 INSTRUCCIONES:')
            print('   1. Si la sesión expiró, inicia sesión.')
            print('   2. Navega semana por semana para capturar los datos.')
            print('   3. Cuando termines, presiona ENTER aquí.')
            input('\n⏳ Presiona ENTER para guardar y cerrar...')

            # Guardar sesión para la próxima vez
            context.storage_state(path=self.session_file)
            print(f'✔️  Sesión guardada → {self.session_file}')

            self._guardar_resultados()
            browser.close()

---
## CELDA 4 — Clase `UberReportExtractor`
**Responsabilidad única:** Recibe la ruta de UN pdf y extrae los datos financieros de esa semana.

> Probado con PDF real: semana 23/12/2024–30/12/2024 ✅

In [6]:
class UberReportExtractor:
    """
    Extrae datos de UN resumen semanal de Uber en formato PDF.
    Uso:
        extractor = UberReportExtractor('ruta/archivo.pdf')
        if extractor.load_pdf():
            extractor.parse_data()
            print(extractor.data)
    """

    def __init__(self, file_path: str):
        self.file_path = file_path
        self.full_text = ''
        self.page1_text = ''
        self.texto_plano = ''
        self.data = {
            'driver_name':              'Roman Yair Ortega',
            'weekly_period':            'Not found',
            'total_trips':              0,
            'connected_hours_decimal':  0.0,
            'gross_amount':             0.0,
            'tips_amount':              0.0,
            'incentives_amount':        0.0,
            'taxes_amount':             0.0,
        }

    @staticmethod
    def _to_float(val_str: str) -> float:
        """
        Convierte strings de dinero a float.
        Soporta: '9.792,29' → 9792.29 / '3,411.47' → 3411.47 / '70,00' → 70.0
        """
        val_str = re.sub(r'[^\d\.,]', '', val_str)
        if not val_str:
            return 0.0
        if ',' in val_str and '.' in val_str:
            if val_str.rfind(',') > val_str.rfind('.'):
                val_str = val_str.replace('.', '').replace(',', '.')
            else:
                val_str = val_str.replace(',', '')
        elif ',' in val_str:
            if len(val_str) - val_str.rfind(',') == 3:
                val_str = val_str.replace(',', '.')
            else:
                val_str = val_str.replace(',', '')
        return float(val_str)

    def load_pdf(self) -> bool:
        """Carga el PDF y construye el texto plano. Devuelve True/False."""
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for i, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean = text.replace('\x00', '')
                        self.full_text += clean + '\n'
                        if i == 0:
                            self.page1_text = clean
            self.texto_plano = ' '.join(self.full_text.split())
            return True
        except Exception as e:
            print(f'❌ Error cargando {self.file_path}: {e}')
            return False

    def parse_data(self):
        """Extrae todos los campos con regex. Rellena self.data."""

        # 1. Periodo semanal (formato: 23/12/2024)
        m = re.search(r'(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})', self.page1_text)
        if m:
            self.data['weekly_period'] = f'Del {m.group(1)} al {m.group(2)}'
        else:
            m2 = re.search(r'(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})', self.page1_text, re.IGNORECASE)
            if m2:
                self.data['weekly_period'] = re.sub(r'\s+\d{1,2}\s+[ap]\.\s+m\.', '', m2.group(1)).strip()

        # 2. Total de viajes
        m = re.search(r'viajes completados[^\d]*(\d+)', self.full_text, re.IGNORECASE)
        if m:
            self.data['total_trips'] = int(m.group(1))

        # 3. Horas conectadas → decimal
        # Formato real del PDF: 'Tiempo de trabajo efectivo = P2+P3 45 h 33 m'
        m = re.search(
            r'Tiempo de trabajo efectivo\s*=?\s*(?:P2\+P3)?\s*(\d+)\s*h\s*(\d+)\s*m',
            self.texto_plano, re.IGNORECASE
        )
        if m:
            self.data['connected_hours_decimal'] = round(int(m.group(1)) + int(m.group(2)) / 60.0, 2)

        # 4. Monto bruto
        m = re.search(
            r'(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[,\.]\d{2})',
            self.texto_plano, re.IGNORECASE
        )
        if m:
            self.data['gross_amount'] = self._to_float(m.group(1))

        # 5. Propinas
        m = re.search(
            r'(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[,\.]\d{2})',
            self.texto_plano, re.IGNORECASE
        )
        if m:
            self.data['tips_amount'] = self._to_float(m.group(1))

        # 6. Incentivos / otras ganancias
        m = re.search(
            r'(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[,\.]\d{2})',
            self.texto_plano, re.IGNORECASE
        )
        if m:
            self.data['incentives_amount'] = self._to_float(m.group(1))

        # 7. Impuestos (siempre negativo)
        m = re.search(
            r'Impuestos\s*-\s*([\d\.,]+[,\.]\d{2})',
            self.texto_plano, re.IGNORECASE
        )
        if m:
            self.data['taxes_amount'] = -abs(self._to_float(m.group(1)))

---
## CELDA 5 — Clase `TripActivity`
**Responsabilidad única:** Recibe UN objeto JSON de viaje y lo convierte en un dict limpio.

Campos del JSON de Uber que maneja:
- `uuid`, `recognizedAt` (Unix timestamp), `activityTitle`, `formattedTotal`
- `tripMetaData.formattedDuration`, `formattedDistance`, `pickupAddress`, `dropOffAddress`

In [7]:
class TripActivity:
    """
    Parsea UN viaje individual del JSON de getWebActivityFeed.
    Convierte strings de Uber a tipos numéricos limpios.
    Uso:
        viaje = TripActivity(raw_json_object)
        if viaje.is_valid:
            print(viaje.data)
    """

    def __init__(self, raw_data: dict):
        self.raw = raw_data
        self.data = {}
        self._parse()

    @staticmethod
    def _parse_money(text: str) -> float:
        """'45,65 MXN' → 45.65"""
        if not text:
            return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', text)
        if ',' in cleaned and '.' in cleaned:
            if cleaned.rfind(',') > cleaned.rfind('.'):
                cleaned = cleaned.replace('.', '').replace(',', '.')
            else:
                cleaned = cleaned.replace(',', '')
        elif ',' in cleaned:
            if len(cleaned) - cleaned.rfind(',') == 3:
                cleaned = cleaned.replace(',', '.')
            else:
                cleaned = cleaned.replace(',', '')
        return float(cleaned) if cleaned else 0.0

    @staticmethod
    def _parse_duration(text: str) -> float:
        """'15 min 12 seg' → 15.20 (minutos decimales)"""
        if not text:
            return 0.0
        minutes = re.search(r'(\d+)\s*min', text)
        seconds = re.search(r'(\d+)\s*seg', text)
        m = int(minutes.group(1)) if minutes else 0
        s = int(seconds.group(1)) if seconds else 0
        return round(m + s / 60.0, 2)

    @staticmethod
    def _parse_distance(text: str) -> float:
        """'7.10 km' → 7.10  /  '2,30 km' → 2.30"""
        if not text:
            return 0.0
        m = re.search(r'([\d\.,]+)', text)
        return float(m.group(1).replace(',', '.')) if m else 0.0

    @staticmethod
    def _parse_timestamp(ts) -> str:
        """1744010733 → '2025-04-07 10:25:33'"""
        try:
            return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
        except:
            return 'Unknown'

    def _parse(self):
        meta = self.raw.get('tripMetaData') or {}
        fare = self._parse_money(self.raw.get('formattedTotal', ''))
        dist = self._parse_distance(meta.get('formattedDistance', ''))
        dur  = self._parse_duration(meta.get('formattedDuration', ''))

        self.data = {
            'trip_id':         self.raw.get('uuid', ''),
            'trip_datetime':   self._parse_timestamp(self.raw.get('recognizedAt', 0)),
            'service_type':    self.raw.get('activityTitle', ''),
            'status':          self.raw.get('status', ''),
            'pickup_address':  meta.get('pickupAddress', ''),
            'dropoff_address': meta.get('dropOffAddress', ''),
            'distance_km':     dist,
            'duration_min':    dur,
            'fare_amount':     fare,
            'pesos_por_km':    round(fare / dist, 2) if dist > 0 else 0.0,
            'pesos_por_min':   round(fare / dur, 2)  if dur > 0  else 0.0,
        }

    @property
    def is_valid(self) -> bool:
        """True si es un viaje COMPLETADO con distancia válida."""
        return (
            self.raw.get('type') == 'TRIP' and
            self.raw.get('status') == 'COMPLETED' and
            self.data['distance_km'] > 0
        )

---
## CELDA 6 — Clase `UberDatasetManager`
**Responsabilidad:** Orquesta ambas fuentes. Carga PDFs y JSON, fusiona y calcula columnas derivadas.

In [8]:
class UberDatasetManager:
    """
    Orquesta la construcción del dataset maestro.
    Uso:
        manager = UberDatasetManager(
            pdf_folder='Descargas_Uber_Marzo_2025',
            json_file='viajes_uber_detallados.json'
        )
        manager.build()
        df_semanas = manager.df_weekly
        df_viajes  = manager.df_trips
    """

    def __init__(self, pdf_folder: str, json_file: str):
        self.pdf_folder = pdf_folder
        self.json_file  = json_file
        self.df_weekly  = None
        self.df_trips   = None

    def _load_pdf_reports(self) -> pd.DataFrame:
        pdf_files = glob.glob(os.path.join(self.pdf_folder, '*.pdf'))
        print(f'   PDFs encontrados: {len(pdf_files)}')
        records = []
        for fp in pdf_files:
            ext = UberReportExtractor(fp)
            if ext.load_pdf():
                ext.parse_data()
                records.append(ext.data)
        if not records:
            print('   ⚠️  Sin datos en PDFs.')
            return pd.DataFrame()
        df = pd.DataFrame(records)
        df = df[df['total_trips'] > 0].copy()
        df['net_earnings']  = df['gross_amount'] + df['taxes_amount']
        df['pesos_por_hora'] = (df['net_earnings'] / df['connected_hours_decimal']).round(2)
        print(f'   ✅ Semanas cargadas: {len(df)}')
        return df

    def _load_trip_activities(self) -> pd.DataFrame:
        if not os.path.exists(self.json_file):
            print(f'   ⚠️  No se encontró: {self.json_file}')
            return pd.DataFrame()
        with open(self.json_file, 'r', encoding='utf-8') as f:
            raw_list = json.load(f)
        print(f'   Registros en JSON: {len(raw_list)}')
        trips = [TripActivity(r).data for r in raw_list if TripActivity(r).is_valid]
        if not trips:
            print('   ⚠️  Sin viajes válidos en JSON.')
            return pd.DataFrame()
        df = pd.DataFrame(trips)
        df['trip_datetime'] = pd.to_datetime(df['trip_datetime'])
        df = df.sort_values('trip_datetime').reset_index(drop=True)
        print(f'   ✅ Viajes cargados: {len(df)}')
        return df

    def build(self):
        """Construye ambos DataFrames. Punto de entrada principal."""
        print('=' * 55)
        print('⚙️  CONSTRUYENDO DATASET MAESTRO...')
        print('=' * 55)
        print('\n📄 Fuente 1: PDFs semanales')
        self.df_weekly = self._load_pdf_reports()
        print('\n🌐 Fuente 2: JSON de viajes')
        self.df_trips  = self._load_trip_activities()
        print('\n' + '=' * 55)
        print('🏆 Dataset listo.')
        if self.df_weekly is not None and not self.df_weekly.empty:
            print(f'   Semanas : {len(self.df_weekly)} | '
                  f'Viajes (PDF): {self.df_weekly["total_trips"].sum()} | '
                  f'Neto total: ${self.df_weekly["net_earnings"].sum():,.2f} MXN')
        if self.df_trips is not None and not self.df_trips.empty:
            print(f'   Viajes (JSON): {len(self.df_trips)} | '
                  f'Km totales: {self.df_trips["distance_km"].sum():,.1f} km')

---
## CELDA 7 — Clase `DataExporter`
**Responsabilidad única:** Guarda los DataFrames como CSVs con validación.

In [9]:
class DataExporter:
    """
    Guarda los DataFrames del Manager como CSVs limpios.
    Uso:
        exporter = DataExporter(output_folder='dataset_output')
        exporter.save_all(manager)
    """

    def __init__(self, output_folder: str = 'dataset_output'):
        self.output_folder = output_folder
        os.makedirs(output_folder, exist_ok=True)

    def save(self, df: pd.DataFrame, filename: str) -> bool:
        if df is None or df.empty:
            print(f'   ⚠️  [{filename}] vacío, no se guardó.')
            return False
        path = os.path.join(self.output_folder, filename)
        df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f'   💾 {filename} → {len(df)} filas × {len(df.columns)} cols → {path}')
        return True

    def save_all(self, manager: UberDatasetManager):
        print('\n💾 EXPORTANDO...')
        self.save(manager.df_weekly, 'dataset_semanal.csv')
        self.save(manager.df_trips,  'dataset_viajes.csv')
        print(f'   ✅ Archivos en: {self.output_folder}/')

---
## CELDA 8 — Ejecución del Pipeline

**Orden de ejecución recomendado:**
1. Corre `UberPDFDownloader` para descargar los PDFs del mes
2. Corre `UberActivityBot` para capturar los viajes del JSON
3. Corre el pipeline de abajo para construir el dataset maestro

In [ ]:
# ============================================================
# PASO 1 — Descargar PDFs
# ============================================================
# OPCIÓN A: Descargar un solo mes (pide mes/año por consola)
# downloader = UberPDFDownloader()
# await downloader.run()

# OPCIÓN B: Descargar varios meses en secuencia
# downloader = UberPDFDownloader()
# meses_a_descargar = [
#     ('diciembre', '2024'),
#     ('enero',     '2025'),
#     ('febrero',   '2025'),
#     ('marzo',     '2025'),
#     ('abril',     '2025'),
# ]
# for mes, anio in meses_a_descargar:
#     await downloader.run(mes=mes, anio=anio)
# print(f'🏆 Total descargado: {downloader.downloads_count} PDFs → Uber_Reports/')

# ============================================================
# PASO 2 — Capturar viajes JSON
# ============================================================
# activity_bot = UberActivityBot()
# activity_bot.run()

# ============================================================
# PASO 3 — Construir el dataset maestro
# ⚙️  Apunta a la carpeta raíz Uber_Reports/ para procesar TODOS los meses
#     o a una subcarpeta específica para procesar solo un mes
# ============================================================
PDF_FOLDER = 'Uber_Reports/Abril_2025'         # Un mes específico
# PDF_FOLDER = 'Uber_Reports'                  # ← Todos los meses juntos (si quieres)
JSON_FILE  = 'viajes_uber_detallados.json'
OUTPUT_DIR = 'dataset_output'

manager  = UberDatasetManager(pdf_folder=PDF_FOLDER, json_file=JSON_FILE)
manager.build()

exporter = DataExporter(output_folder=OUTPUT_DIR)
exporter.save_all(manager)

# ============================================================
# PASO 4 — Vista previa
# ============================================================
if manager.df_weekly is not None and not manager.df_weekly.empty:
    print('\n📊 DATASET SEMANAL:')
    display(manager.df_weekly.style.format({
        'gross_amount':            '${:,.2f}',
        'tips_amount':             '${:,.2f}',
        'incentives_amount':       '${:,.2f}',
        'taxes_amount':            '${:,.2f}',
        'net_earnings':            '${:,.2f}',
        'connected_hours_decimal': '{:.2f} hrs',
        'pesos_por_hora':          '${:,.2f}',
    }))

if manager.df_trips is not None and not manager.df_trips.empty:
    print('\n🗺️  PRIMEROS 10 VIAJES:')
    display(manager.df_trips.head(10).style.format({
        'fare_amount':    '${:,.2f}',
        'distance_km':   '{:.2f} km',
        'duration_min':  '{:.1f} min',
        'pesos_por_km':  '${:.2f}',
        'pesos_por_min': '${:.2f}',
    }))